# Notebook 30 — Cluster-level mass conservation: proposing and testing a law of collapse allocation

Hypothesis under test (stated before analysis): pruning-induced collapse is governed at the level of
confusability clusters — total recall mass lost per cluster is stable across independently trained
baseline-mask pairs, while the allocation of that loss among cluster members is seed-dependent
symmetry breaking. Evidence motivating the hypothesis: stable aggregate prevalence with seed-dependent
victim identity (paired-seed analysis), near-chance strict pairwise probe AUC for protocol-matched
DoS/DDoS siblings, and mask Jaccard overlap of only 11-12%.

The confusable clusters are fixed a priori from the strict pairwise probe evidence: the four
protocol-matched flood pairs. All statistics are computed on the committed paired-seed result tables;
no model is loaded and no experiment is rerun. The decisive test is exact: the eight flood classes
admit exactly 105 perfect pairings, so the null distribution is enumerated, not sampled.

This notebook reports the outcome of the test whichever way it falls.

In [ ]:
# --- Colab bootstrap ---
try:
    from google.colab import drive; drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
except Exception:
    REPO = '.'
import os, sys
os.chdir(REPO); sys.path.insert(0, REPO)
import numpy as np, pandas as pd
from src.config import PATHS
OUT = PATHS.tables('comnet')
print('repo:', REPO)

In [ ]:
# Inputs: committed paired-seed effects, per-class test supports, family mapping
eff = pd.read_csv(OUT / 'paired_seed_per_class_effects.csv')
sup = pd.read_csv(OUT / 'bn_control_M0_per_class.csv').set_index('label')['support']
fam = pd.read_csv(OUT / 'ciciot2023_alert_family_mapping.csv').set_index('fine_label')['alert_family']

assert set(['seed','cell','class','recall_loss']).issubset(eff.columns)
p80 = eff[eff.cell == 'prune80'].copy()
p80['mass'] = p80['recall_loss'] * p80['class'].map(sup)
assert p80['mass'].notna().all() and len(p80) == 170
M = p80.pivot(index='class', columns='seed', values='mass')
assert M.shape == (34, 5)

# A-priori confusable clusters: the four protocol-matched flood pairs (strict pairwise probe evidence)
PAIRS = [('DoS-SYN_Flood','DDoS-SYN_Flood'), ('DoS-TCP_Flood','DDoS-TCP_Flood'),
         ('DoS-UDP_Flood','DDoS-UDP_Flood'), ('DoS-HTTP_Flood','DDoS-HTTP_Flood')]
EIGHT = [c for p in PAIRS for c in p]
for c in EIGHT: assert c in M.index
print('inputs verified:', M.shape, '| clusters:', len(PAIRS), 'pairs')

In [ ]:
# T1 - Compensation signature per pair.
# Var(A+B) < Var(A)+Var(B) iff Cov(A,B) < 0: one sibling absorbs when the other collapses.
rows = []
for a, b in PAIRS:
    vr = np.var(M.loc[a]+M.loc[b], ddof=1) / (np.var(M.loc[a], ddof=1) + np.var(M.loc[b], ddof=1))
    rows.append({'pair': a.replace('_Flood','') + '+' + b.replace('_Flood',''),
                 'variance_ratio': round(float(vr), 4),
                 'signature': 'compensating' if vr < 1 else 'co-moving'})
t1 = pd.DataFrame(rows)
print(t1.to_string(index=False))

In [ ]:
# T2 - Exact permutation test. Statistic: total pair-mass variance across seeds.
# Null: all 105 perfect pairings of the eight flood classes. Law predicts the true
# protocol pairing minimises the statistic (low exact p).
def matchings(items):
    if not items:
        yield []; return
    a = items[0]
    for i in range(1, len(items)):
        for rest in matchings(items[1:i] + items[i+1:]):
            yield [(a, items[i])] + rest

def stat(pairing):
    return sum(np.var(M.loc[a] + M.loc[b], ddof=1) for a, b in pairing)

null = np.array([stat(m) for m in matchings(EIGHT)])
obs = float(stat(PAIRS))
p_exact = float((null <= obs).mean())
assert len(null) == 105
print(f'observed total pair-mass variance: {obs:,.0f}')
print(f'null over 105 pairings: min {null.min():,.0f}, median {np.median(null):,.0f}, max {null.max():,.0f}')
print(f'exact p = {p_exact:.4f}')

In [ ]:
# T3 - Variance decomposition of per-class recall loss (the eight flood classes),
# per-seed global level removed. Law predicts within-pair share dominates.
L = p80[p80['class'].isin(EIGHT)].pivot(index='class', columns='seed', values='recall_loss')
Ls = L - L.mean(axis=0)
total = float((Ls**2).sum().sum())
within = 0.0
for a, b in PAIRS:
    pm = (Ls.loc[a] + Ls.loc[b]) / 2
    within += float(((Ls.loc[a] - pm)**2 + (Ls.loc[b] - pm)**2).sum())
within_share = within / total
print(f'within-pair share of loss variance: {within_share:.1%}')

In [ ]:
# Verdict record - written whichever way the tests fall.
verdict = pd.DataFrame([
 {'test': 'T1_compensation_pairs', 'value': int((t1.variance_ratio < 1).sum()), 'of': len(PAIRS)},
 {'test': 'T2_exact_p', 'value': round(p_exact, 4), 'of': 105},
 {'test': 'T3_within_pair_share', 'value': round(within_share, 4), 'of': 1},
])
supported = (p_exact < 0.05) and (within_share > 0.6)
verdict['strong_law_supported_at_n5'] = supported
print(verdict.to_string(index=False))
print()
print('strong cluster-mass-conservation law supported at n=5 seeds:', supported)
print('n=5 gives limited power for variance statistics; non-support at this n does not refute the law.')
print('Decisive follow-ups: additional independent paired seeds, and causal intervention')
print('(sibling-merge retraining; synthetic confusability control).')

t1.to_csv(OUT / 'law_test_pair_compensation.csv', index=False)
verdict.to_csv(OUT / 'law_test_verdict.csv', index=False)
pd.DataFrame({'null_total_pair_variance': null}).to_csv(OUT / 'law_test_permutation_null.csv', index=False)
print()
print('saved: law_test_pair_compensation.csv, law_test_verdict.csv, law_test_permutation_null.csv')

In [ ]:
# --- Commit + push (identity set explicitly; outputs stripped before add) ---
import subprocess, shutil, glob, json as _json
# --- Branch guard: this notebook belongs to the diagnostic paper on main, not the SABER branch ---
_b = subprocess.run(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert _b == 'main', f'checked-out branch is {_b!r}; run `git checkout main` in the repo first (SABER work stays on saber-ids-method)'
subprocess.run(['git', 'config', '--global', 'user.name', 'Md Anas Biswas'], check=True)
subprocess.run(['git', 'config', '--global', 'user.email', 'anasbiswas@gmail.com'], check=True)
cred = '/content/drive/MyDrive/IoT_Trust_Research/.git-credentials'
if os.path.exists(cred):
    shutil.copy(cred, '/root/.git-credentials')
    subprocess.run(['git', 'config', '--global', 'credential.helper', 'store'], check=True)
# strip outputs from THIS notebook only, then stage ONLY this notebook's own files
_own = 'notebooks/30_cluster_mass_conservation_law.ipynb'
if os.path.exists(_own):
    d = _json.load(open(_own))
    for c in d.get('cells', []):
        if c.get('cell_type') == 'code': c['outputs'] = []; c['execution_count'] = None
    _json.dump(d, open(_own, 'w'), indent=1)
subprocess.run(['git', 'add', _own] + glob.glob('results/tables/comnet/law_test_*.csv'), check=True)
r = subprocess.run(['git', 'commit', '-m',
    'notebook 30: cluster mass-conservation law - a-priori test on paired seeds (exact permutation, compensation, variance decomposition)'],
    capture_output=True, text=True)
print(r.stdout or r.stderr)
p = subprocess.run(['git', 'push'], capture_output=True, text=True)
print(p.stderr or p.stdout or 'pushed')
print(subprocess.run(['git', 'log', '--oneline', '-2'], capture_output=True, text=True).stdout)